In [8]:
import torch 
import torch.nn as nn
import torch.nn.functional as F


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

using device: cuda


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_heads):
        super().__init__()
        self.d_heads = d_heads
        self.head_dim = d_model // d_heads
        
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.fc = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        B, T, C = x.shape
        
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)
        
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        
        out = weights @ V
        out = out.transpose(1, 2).reshape(B, T, C)
        
        return self.fc(out)
        

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )
        
    def forward(self, x):
        return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_head)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        
        return x
        
        